# SQLite with Python

SQLite is a file-based relational database — no server needed. It's built into Python via the `sqlite3` module.

Topics:
- Creating a database and tables
- CRUD: INSERT, SELECT, UPDATE, DELETE
- Querying with WHERE, ORDER BY, GROUP BY, HAVING
- JOIN operations
- Pandas ↔ SQLite integration
- Parameterised queries (SQL injection prevention)

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = '/tmp/bookstore.db'
Path(DB_PATH).unlink(missing_ok=True)  # fresh start

# Connect (creates file if it doesn't exist)
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row   # rows behave like dicts
cursor = conn.cursor()
print('Connected to', DB_PATH)

Connected to /tmp/bookstore.db


## 1. Creating Tables

In [2]:
# Create authors table
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS authors (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT NOT NULL,
        country TEXT,
        born    INTEGER
    );

    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT NOT NULL,
        author_id   INTEGER REFERENCES authors(id),
        genre       TEXT,
        pages       INTEGER,
        price       REAL,
        published   TEXT,
        in_stock    INTEGER DEFAULT 1  -- 1=True, 0=False
    );

    CREATE TABLE IF NOT EXISTS sales (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        book_id     INTEGER REFERENCES books(id),
        quantity    INTEGER,
        sale_date   TEXT
    );
''')
conn.commit()
print('Tables created')

Tables created


## 2. INSERT — Adding Data

In [3]:
# Insert with parameterised queries (ALWAYS use ? placeholders — never f-strings)
authors_data = [
    ('George Orwell',     'UK',    1903),
    ('J.K. Rowling',      'UK',    1965),
    ('Yuval Noah Harari', 'Israel',1976),
    ('Malcolm Gladwell',  'Canada',1963),
    ('Agatha Christie',   'UK',    1890),
]
cursor.executemany('INSERT INTO authors (name, country, born) VALUES (?, ?, ?)', authors_data)

books_data = [
    ('1984',                        1, 'Dystopian',   328, 12.99, '1949-06-08', 1),
    ('Animal Farm',                 1, 'Satire',      112,  8.99, '1945-08-17', 1),
    ('Harry Potter and the SS',     2, 'Fantasy',     309, 14.99, '1997-06-26', 1),
    ('Harry Potter and the CoS',    2, 'Fantasy',     341, 14.99, '1998-07-02', 1),
    ('Sapiens',                     3, 'Non-Fiction', 443, 16.99, '2011-01-01', 1),
    ('Homo Deus',                   3, 'Non-Fiction', 450, 16.99, '2015-09-04', 1),
    ('Outliers',                    4, 'Non-Fiction', 309, 13.99, '2008-11-18', 1),
    ('The Tipping Point',           4, 'Non-Fiction', 301, 12.99, '2000-03-01', 1),
    ('Murder on the Orient Express',5, 'Mystery',     256, 11.99, '1934-01-01', 1),
    ('And Then There Were None',    5, 'Mystery',     272, 11.99, '1939-11-06', 0),
]
cursor.executemany(
    'INSERT INTO books (title, author_id, genre, pages, price, published, in_stock) VALUES (?,?,?,?,?,?,?)',
    books_data
)

# Sales data
import random
random.seed(42)
sales_data = [(random.randint(1,10), random.randint(1,50), f'2024-{m:02d}-{d:02d}')
              for m in range(1,7) for d in [5,12,20,28]]
cursor.executemany('INSERT INTO sales (book_id, quantity, sale_date) VALUES (?,?,?)', sales_data)

conn.commit()
print('Data inserted')

Data inserted


## 3. SELECT — Reading Data

In [4]:
# Basic SELECT
print('=== All Authors ===')
for row in cursor.execute('SELECT * FROM authors'):
    print(dict(row))

print('\n=== Books under $14 ===')
for row in cursor.execute('SELECT title, price FROM books WHERE price < 14.00 ORDER BY price'):
    print(f"  {row['title']:40s}  ${row['price']:.2f}")

=== All Authors ===
{'id': 1, 'name': 'George Orwell', 'country': 'UK', 'born': 1903}
{'id': 2, 'name': 'J.K. Rowling', 'country': 'UK', 'born': 1965}
{'id': 3, 'name': 'Yuval Noah Harari', 'country': 'Israel', 'born': 1976}
{'id': 4, 'name': 'Malcolm Gladwell', 'country': 'Canada', 'born': 1963}
{'id': 5, 'name': 'Agatha Christie', 'country': 'UK', 'born': 1890}

=== Books under $14 ===
  Animal Farm                               $8.99
  Murder on the Orient Express              $11.99
  And Then There Were None                  $11.99
  1984                                      $12.99
  The Tipping Point                         $12.99
  Outliers                                  $13.99


In [5]:
# Aggregation: GROUP BY + HAVING
print('=== Average price by genre (>1 book) ===')
result = cursor.execute('''
    SELECT genre,
           COUNT(*)       AS book_count,
           AVG(price)     AS avg_price,
           SUM(pages)     AS total_pages
    FROM   books
    GROUP  BY genre
    HAVING COUNT(*) > 1
    ORDER  BY avg_price DESC
''').fetchall()
for row in result:
    print(f"  {row['genre']:15s}  {row['book_count']} books  avg=${row['avg_price']:.2f}")

=== Average price by genre (>1 book) ===
  Non-Fiction      4 books  avg=$15.24
  Fantasy          2 books  avg=$14.99
  Mystery          2 books  avg=$11.99


## 4. JOIN — Combining Tables

In [6]:
# INNER JOIN: books with their author names
print('=== Books with Author Names ===')
result = cursor.execute('''
    SELECT b.title, a.name AS author, b.genre, b.price
    FROM   books b
    JOIN   authors a ON b.author_id = a.id
    ORDER  BY a.name, b.published
''').fetchall()
for row in result:
    print(f"  {row['author']:25s}  {row['title']:35s}  ${row['price']:.2f}")

=== Books with Author Names ===
  Agatha Christie            Murder on the Orient Express         $11.99
  Agatha Christie            And Then There Were None             $11.99
  George Orwell              Animal Farm                          $8.99
  George Orwell              1984                                 $12.99
  J.K. Rowling               Harry Potter and the SS              $14.99
  J.K. Rowling               Harry Potter and the CoS             $14.99
  Malcolm Gladwell           The Tipping Point                    $12.99
  Malcolm Gladwell           Outliers                             $13.99
  Yuval Noah Harari          Sapiens                              $16.99
  Yuval Noah Harari          Homo Deus                            $16.99


In [7]:
# Multi-table join: total sales per author
print('=== Total Units Sold per Author ===')
result = cursor.execute('''
    SELECT a.name AS author,
           COUNT(DISTINCT b.id)  AS books_sold,
           SUM(s.quantity)        AS total_units,
           ROUND(SUM(s.quantity * b.price), 2) AS total_revenue
    FROM   sales s
    JOIN   books   b ON s.book_id = b.id
    JOIN   authors a ON b.author_id = a.id
    GROUP  BY a.name
    ORDER  BY total_revenue DESC
''').fetchall()
for row in result:
    print(f"  {row['author']:25s}  {row['total_units']:4d} units  ${row['total_revenue']:8,.2f}")

=== Total Units Sold per Author ===
  J.K. Rowling                165 units  $2,473.35
  George Orwell               159 units  $1,633.41
  Yuval Noah Harari            75 units  $1,274.25
  Agatha Christie              94 units  $1,127.06
  Malcolm Gladwell             22 units  $  307.78


## 5. UPDATE and DELETE

In [8]:
# UPDATE: mark out-of-stock books with price reduction
print('Before:', cursor.execute('SELECT title, price, in_stock FROM books WHERE in_stock=0').fetchall())

cursor.execute('''
    UPDATE books
    SET    price = price * 0.90,
           in_stock = 1
    WHERE  in_stock = 0
''')
conn.commit()
print('After:', cursor.execute('SELECT title, price, in_stock FROM books WHERE title=?',
                               ("And Then There Were None",)).fetchone()['price'])

# DELETE: remove old sales records
cursor.execute("DELETE FROM sales WHERE sale_date < '2024-03-01'")
conn.commit()
print(f'Remaining sales: {cursor.execute("SELECT COUNT(*) FROM sales").fetchone()[0]}')

Before: [<sqlite3.Row object at 0x7417ba01b760>]
After: 10.791
Remaining sales: 16


## 6. Pandas ↔ SQLite Integration

In [9]:
# Read query result directly into a DataFrame
df = pd.read_sql('''
    SELECT b.title, a.name AS author, b.genre, b.price,
           COALESCE(SUM(s.quantity), 0) AS total_sold
    FROM   books b
    JOIN   authors a ON b.author_id = a.id
    LEFT JOIN sales s ON b.id = s.book_id
    GROUP  BY b.id
    ORDER  BY total_sold DESC
''', conn)

print(df.to_string(index=False))
print('\nAverage price:', df['price'].mean().round(2))
print('Best seller:', df.iloc[0]['title'])

                       title            author       genre  price  total_sold
    Harry Potter and the CoS      J.K. Rowling     Fantasy 14.990         111
                        1984     George Orwell   Dystopian 12.990          49
                 Animal Farm     George Orwell      Satire  8.990          48
                   Homo Deus Yuval Noah Harari Non-Fiction 16.990          46
     Harry Potter and the SS      J.K. Rowling     Fantasy 14.990          45
Murder on the Orient Express   Agatha Christie     Mystery 11.990          40
                    Outliers  Malcolm Gladwell Non-Fiction 13.990          22
    And Then There Were None   Agatha Christie     Mystery 10.791          20
                     Sapiens Yuval Noah Harari Non-Fiction 16.990          13
           The Tipping Point  Malcolm Gladwell Non-Fiction 12.990           0

Average price: 13.57
Best seller: Harry Potter and the CoS


In [10]:
# Write a DataFrame to SQLite
new_books = pd.DataFrame({
    'title':     ['Deep Work', 'Atomic Habits'],
    'author_id': [4, 4],
    'genre':     ['Non-Fiction', 'Non-Fiction'],
    'pages':     [304, 320],
    'price':     [14.99, 13.99],
    'published': ['2016-01-05', '2018-10-16'],
    'in_stock':  [1, 1]
})
new_books.to_sql('books', conn, if_exists='append', index=False)
print('Inserted', len(new_books), 'new books')
print('Total books:', cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0])

Inserted 2 new books
Total books: 12


## 7. SQL Injection — Why Use Parameters?

In [11]:
# VULNERABLE: building the query by string interpolation
# Simulates a "search by title" feature that trusts user input directly.
malicious_input = "x' OR '1'='1"   # classic injection payload — always-true condition

vulnerable_query = f"SELECT title, price FROM books WHERE title = '{malicious_input}'"
print('Query actually sent to SQLite:')
print(' ', vulnerable_query)

leaked = cursor.execute(vulnerable_query).fetchall()
print(f"\nAttacker searched for a title that doesn't exist, but got back {len(leaked)} rows")
print('(every row in the table — the WHERE clause was neutralised by the injected OR):')
for row in leaked:
    print(f"  {row['title']:35s}  ${row['price']:.2f}")

Query actually sent to SQLite:
  SELECT title, price FROM books WHERE title = 'x' OR '1'='1'

Attacker searched for a title that doesn't exist, but got back 12 rows
(every row in the table — the WHERE clause was neutralised by the injected OR):
  1984                                 $12.99
  Animal Farm                          $8.99
  Harry Potter and the SS              $14.99
  Harry Potter and the CoS             $14.99
  Sapiens                              $16.99
  Homo Deus                            $16.99
  Outliers                             $13.99
  The Tipping Point                    $12.99
  Murder on the Orient Express         $11.99
  And Then There Were None             $10.79
  Deep Work                            $14.99
  Atomic Habits                        $13.99


In [12]:
# THE FIX: parameterised query — sqlite3 treats the value as pure data, never as SQL syntax
user_input = "x' OR '1'='1"   # same payload as above

cursor.execute('SELECT title, price FROM books WHERE title = ?', (user_input,))
result = cursor.fetchall()
print(f'Parameterised query returned {len(result)} rows for the same malicious input')
print('(the payload is matched literally as a title string, not parsed as SQL — 0 rows expected)')

# A destructive payload is neutralised the same way — the table survives untouched
drop_attempt = "'; DROP TABLE books; --"
cursor.execute('SELECT * FROM books WHERE title = ?', (drop_attempt,))
print('\nBook count after a DROP-TABLE payload through a parameterised query:',
      cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0])

Parameterised query returned 0 rows for the same malicious input
(the payload is matched literally as a title string, not parsed as SQL — 0 rows expected)

Book count after a DROP-TABLE payload through a parameterised query: 12


In [13]:
# Always close the connection when done
conn.close()
print('Connection closed')

Connection closed


## 8. Why an Index Beats a Full Scan — Measured

A `WHERE` lookup on an unindexed column has to check every row — SQLite (and a Python list) both
degrade to **O(n)**. A B-tree index keeps the column's values in sorted order in a balanced tree,
so a lookup follows one root-to-leaf path — **O(log n)** comparisons — instead of scanning
everything. Below: a genuine Python linear scan over a list of 100,000 `(id, value)` tuples,
timed against an SQLite query on the *same* 100,000 rows with a `CREATE INDEX` on `value`.

In [14]:
import time
import random

N = 100_000
random.seed(0)
records = [(i, random.randint(0, N * 10)) for i in range(N)]   # (id, value) pairs
target = records[N // 2][1]                                     # a value guaranteed to exist

# ---- Method 1: manual linear scan over a plain Python list (no index of any kind) ----
def linear_scan(rows, target_value):
    for row_id, value in rows:
        if value == target_value:
            return row_id
    return None

t0 = time.perf_counter()
found_id_scan = linear_scan(records, target)
t_scan = time.perf_counter() - t0

# ---- Method 2: same data in SQLite, WITHOUT an index — a full table scan too ----
idx_conn = sqlite3.connect(':memory:')
idx_conn.execute('CREATE TABLE lookup (id INTEGER, value INTEGER)')
idx_conn.executemany('INSERT INTO lookup VALUES (?, ?)', records)
idx_conn.commit()

t0 = time.perf_counter()
found_id_noidx = idx_conn.execute('SELECT id FROM lookup WHERE value = ?', (target,)).fetchone()[0]
t_sql_noindex = time.perf_counter() - t0

# ---- Method 3: same table, now WITH a B-tree index on `value` ----
idx_conn.execute('CREATE INDEX idx_value ON lookup(value)')
idx_conn.commit()

t0 = time.perf_counter()
found_id_idx = idx_conn.execute('SELECT id FROM lookup WHERE value = ?', (target,)).fetchone()[0]
t_sql_index = time.perf_counter() - t0

plan = idx_conn.execute('EXPLAIN QUERY PLAN SELECT id FROM lookup WHERE value = ?', (target,)).fetchall()

print(f'Rows: {N:,}   target value: {target}\n')
print(f'Python linear scan (list):     {t_scan*1000:8.3f} ms  -> id={found_id_scan}')
print(f'SQLite, NO index (table scan): {t_sql_noindex*1000:8.3f} ms  -> id={found_id_noidx}')
print(f'SQLite, WITH index (B-tree):   {t_sql_index*1000:8.3f} ms  -> id={found_id_idx}')
print(f'\nSpeedup, indexed vs Python linear scan: {t_scan / t_sql_index:,.0f}x')
print(f'Speedup, indexed vs SQLite table scan:   {t_sql_noindex / t_sql_index:,.0f}x')
print(f'\nEXPLAIN QUERY PLAN with the index present: {plan}')

idx_conn.close()

Rows: 100,000   target value: 824501

Python linear scan (list):        0.845 ms  -> id=50000
SQLite, NO index (table scan):    2.403 ms  -> id=50000
SQLite, WITH index (B-tree):      0.082 ms  -> id=50000

Speedup, indexed vs Python linear scan: 10x
Speedup, indexed vs SQLite table scan:   29x

EXPLAIN QUERY PLAN with the index present: [(3, 0, 62, 'SEARCH lookup USING INDEX idx_value (value=?)')]


## Quick Summary

| Operation | SQL Snippet |
|-----------|-------------|
| Create table | `CREATE TABLE t (id INTEGER PRIMARY KEY, ...)` |
| Insert | `INSERT INTO t (col1) VALUES (?)` with params |
| Select | `SELECT col FROM t WHERE cond ORDER BY col LIMIT n` |
| Aggregate | `SELECT col, COUNT(*), AVG(x) FROM t GROUP BY col HAVING ...` |
| Join | `SELECT ... FROM t1 JOIN t2 ON t1.id = t2.fk` |
| Update | `UPDATE t SET col = val WHERE cond` |
| Delete | `DELETE FROM t WHERE cond` |
| Pandas read | `pd.read_sql('SELECT ...', conn)` |
| Pandas write | `df.to_sql('table', conn, if_exists='append')` |

> Always use `?` placeholders for user-supplied values — never f-strings.

**Next →** [08 – EDA Projects](../08-eda-projects/)